Introduciamo il middleware orientato ai messaggi (**MOM Message Oriented Middleware**) partendo da un limite di RPC: con una RPC, $A$ invoca un metodo offerto da $B$. Anche se la chiamata è asincrona e $A$ può continuare a fare altro, purché l'operazione vada a buon fine è necessario *che $B$ sia raggiungibile e in grado di riceverla **durante** la chiamata!*

Con **MOM** invece $A$ può depositare un messaggio del tipo "elabora questa foto" e il middleware lo conserva per un certo periodo, per cui $B$ potrà recuperarlo quando lui sarà disponibile. Si parla in questo senso di **comunicazione persistente**, e in termini di decoupling ricordiamo:
- **Decoupling di Sincronizzazione**: *chi invia deve restare bloccato mentre lavora?* MOM lo soddisfa perché $A$, nelle implementazioni asincrone, può continuare a fare altro dopo aver affidato il msg al middleware. *Questo è il decoupling che garantiva anche RPC asincrono*
- **Decoupling temporale**: *mittente e destinatario devono essere attivi nello stesso momento?* MOM lo garantisce grazie alla comunicazione persistente, RPC no
- **Spaziale**: *il mittente deve sapere quale istanza riceverà il messaggio?* Abbiamo visto che in RPC tipicamente il client deve conoscere l'endpoint del server se non c'è un meccanismo di service discovery, mentre in MOM $A$ invia semplicemente alla coda o al topic; è il middleware a gestire la consegna ai destinatari!

Chiaramente, RPC e messaggistica possono convivere nella stessa applicazione, es. un microservizio può usare una RPC per chiedere un dato che gli serve subito e affidare a una coda un lavoro che può essere invece svolto più tardi...

### Message Queue Pattern
Una **coda di messaggi** raccoglie i lavori inviati da dei **producer**. Uno o più **consumer** li prelevano per svolgerli. 

<img src="imgs/7_0.png" width="400">

es. tre worker in grado di ridimensionare una foto. Il producer inserisce in coda un msg per ogni foto --> quando un worker è libero, può prelevare il lavoro. In questo modo il msg relativo a una foto può essere prelevato da worker diversi: in questo senso le code di messaggi si usano per fare **load balancing** tra più istanze di un servizio.  
Altri use case sono *task scheduling*, *logging*, *tracing*...

Questo pattern segue una comunicazione **one-to-one**: ogni msg è prelevato da un **singolo consumer**, anche se possono esestere più consumer che potrebbero prelevarlo.

Tra le operazioni tipiche di una coda di messaggi:
- `put`: il producer mette un messaggio nella coda e può proseguire senza aspettare che il consumer lo elabori
- `get`: il consumer aspetta finché non può recuperare il msg (sincrono). Alcuni sistemi permettono di richiedere un msg che però rispetti certi criteri
- `poll`: il consumer controlla se c'è messaggio. Se la coda è vuota --> riprende subito il proprio lavoro.
- `notify`: il consumer registra un *handler*; quando il msg è disponibile, il middleware lo notifica di modo che il consumer possa reagire.

### Pub/Sub Pattern
Qui cambiamo esigenza: una foto appena caricata interessa a **più componenti** (es. uno deve creare un'anteprima, uno deve aggiornare un indice..). Vogliamo che **entrambi** siano informati dello stesso evento.

Nel pattern pub/sub, il **publisher** pubblica il msg su un **topic** (es. `foto-caricata`) e i **subscriber** esprimono interesse **sottoscrivendo quel topic**. Il middleware consulta le sottoscrizioni e recapita l'evento a tutte quelle che corrispondono: qui si parla di comunicazione **one-to-many**.

Quindi in generale se c'è un lavoro da svolgere e basta che lo faccia un worker --> queue; altrimenti se è accaduto qualcosa e tutti i componenti interessati devono esserne informati --> pub/sub.  

Il componente che si occupa di abbinare messaggi e sottoscrizioni si chiama **event dispatcher**. Questo può considerare anche eventuali **filtri**, ad es. un subscriber può voler ricevere solo un certo tipo di evento dentro un topic. Il dispatcher è un ruolo logico: in un sistema grande è tipicamente necessario che sia realizzato in modo distribuito per evitare SPOF e colli di bottiglia.

Tra le operazioni (API) tipiche: 
- **publish(event)**
- **subscribe(filter, notify_cb, expiry) -> sub_handle**: il subscriber si registra per ricevere eventi di un certo topic, eventualmente indicando un filtro, una callback (notify_cb) e una scadenza.  
Riceve in output un *handle* che può usare per annullare la sottoscrizione tramite **unsubscribe(sub_handle)**.  
Quando un evento corrisponde, il middleware può invocare la callback **notify_cb(sub_handle, event)**.  

### Semantiche e modelli di consegna MOM
Ma come si gestiscono i fallimenti? Immaginiamo un messaggio che contenga l'istruzione "incrementa il contatore della parola `casa`":
- **At-most-once** == **zero oppure una consegna**: il msg può andare perso, ma se arriva il sistema offre comunque la garanzia di consegnarlo al più una volta.
- **At-least-once** == **una o più consegne**, entro le condizioni previste dal sistema. Un meccanismo tipico è far inviare al consumer una conferma (**ACK**), e se il middleware non riceve la conferma può allora riconsegnare il messaggio.. tuttavia il consumer potrebbe aver già incrementato il contatore, da qui nasce il problema dei duplicati
- **Exactly-once** == il msg produce **un solo effetto logico**: qui è da distinguere tra consegna e elaborazione; il msg può anche arrivare due volte, ma il secondo arrivo deve essere gestito affinché vi sia alla fine dei conti un solo incremento. Si deve quindi avere garanzia che anche dopo un guasto ci siano le info necessarie per capire ciò che è stato già elaborato...

Un modo per rendere sicura la ripetizione di un'operazione potrebbe essere quella di assegnare ad ogni msg un ID univoco e ricordare quelli già elaborati (scarto i duplicati), oppure nel caso del contatore invece di dire "incrementa di 1" si può dire "fai il set a tot valore" (es. "fai il set a 5"), trasformando l'operazione in un'**operazione idempotente**.

Altre due semantiche di consegna specifiche di MOM sono:
- **Timeout-based delivery**: garantisce semantica **at-least-once**, senza che però un singolo consumer possa processare più volte il messaggio (at least once perché consumer diversi potrebbero processarlo).  
Il consumer $A$ riceve il msg --> il middleware lo conserva, ma lo rende **invisibile** per un certo periodo di tempo (timeout, es. 60 sec). Durante quel periodo gli altri consumer non possono prelevare il msg.  
Se $A$ termina e invia l'ACK entro il tiemout, il middleware elimina il msg, altrimenti allo scadere del timeout il msg torna disponibile per un altro tentativo
-**Transaction-based delivery**: garantisce che i msg siano cancellati dalla coda solo dopo che sono stati ricevuti e processati dal consumatore.  
L'idea è quindi collegare l'effetto dell'elaborazione e la conferma definitiva del consumo: se la transazione riesce, il *commit* rende **definitivo** il risultato e il sistema può considerare il msg consumato.  
Se fallisce, il *rollback* annulla le modifiche comprese nella transazione e il msg resta recuperabile.  
Questa fgaranzia vale per le operazioni che partecipano alla specifica transazione, ricordiamo questa semantica perché torna tra le configurazioni avanzate di Kafka.

<img src="imgs/7_1.png" width="400">

---

Mentre la semantica di consegna ci dice **quante volte** un msg può arrivare, il **modello di consegna** ci dice **chi prende l'iniziativa**:
- **Push**: il middleware segnala o consegna un msg quando è disponibile --> il consumer può attendere la notifica e reagire quando arriva
- **Pull**: è il consumer a richiedere al middleware se ci sono dati da leggere, ad es. tramite polling. Lo fa quando è pronto ad elaborarne altri.

Vedremo come Kafka usi **pull per i consumer**: il vantaggio è che ciascun consumer può recuperare i dati secondo il proprio ritmo e il middleware risponde alle richieste di lettura. Lo svantaggio è quando la coda è vuota e il consumer interroga troppo spesso il sistema.

### Routing e trasformazione dei msg
Finora abbaimo immaginato una coda facilmente raggiungibile, tuttavia in alcuni sistemi reali le code sono **distribuite**. Immaginiamo un modello dove il mittente deposita il msg nella propria **coda locale** e il destinatario la recupera dalla sua. In mezzo, dei **queue manager** lo inoltrano anche attraverso nodi intermedi.

<img src="imgs/7_2.png" width="400">

In questo senso i queue manager formano una **rete overlay**: ci ricolleghiamo quindi la routing già studiato --> diverse possibilità di instradamento come usare **tabelle di routing** (in una **rete statica**, sono modificate manualmente, in una **dinamica** vengono aggiornate mentre il sistema è in funzione, runtime --> più complesse ma supportano routing scalabile e adattivo), algoritmi di **flooding** etc..

Un altro problema è che il msg può arrivare al componente giusto, ma **in un formato che questo non è in grado di leggere** (es. il producer invia un campo chiamato `costumer_id` mentre un'applicazione già esistente si aspetta `cliente`) --> si può adottare un **broker pattern** dove il **message broker** può applicare regole di conversione e anche scegliere la destinazione in base al contenuto o al tipo di msg.  
Chiaramente anche qui se tutto passasse attraverso un solo broker SPOF e collo di bottiglia --> tipicamente implementato in modo distribuito.

**Coda** e **sub/pub** descrivono il modo in cui vogliamo distribuire i msg. Uno stesso prodotto può offrire entrambi i comportamenti, ad es. **RabbitMQ**, **Kafka**, **Pulsar**, **Redis** e **IBM MQ**. Si distinguono questi strumenti dai **servizi gestiti in cloud**, dove il provider si occupa di gestire buona parte dell'infrastruttura.

Amazon SQS per rivedere un servizio reale di timeout di visibilità e poi Kafka come esempio di sistema distribuito.